# 🗂️ Bloque 1 — Sesión 2: Formatos de almacenamiento, carga de datos y repositorios

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~4 horas.

---

### Contenidos de esta sesión
- **1.3** Formatos de almacenamiento habituales: CSV, JSON, XML, Excel y bases de datos relacionales
- **1.4** Carga de datos con Python/pandas: `read_csv`, `read_json`, `read_sql`, `read_excel`
- **1.5** Repositorios públicos de datos para ML: Kaggle, UCI Machine Learning Repository, Hugging Face Datasets

### Criterio de evaluación asociado
> **(CE a)** Se han descrito técnicas y posibilidades de almacenaje de los datos para componer conjuntos, identificando su naturaleza.

In [ ]:
# Instalaciones y librerías necesarias (ejecutar una sola vez)
import pandas as pd
import json
import xml.etree.ElementTree as ET
import sqlite3
import os

print('✅ Librerías cargadas correctamente')

---
## 1.3 Formatos de almacenamiento habituales

Los datos en proyectos de ML llegan en muy distintos formatos. Conocer sus características permite elegir el más adecuado para cada caso y saber cómo cargarlos correctamente.

| Formato | Extensión | Tipo | Ventajas | Limitaciones |
|---------|-----------|------|----------|--------------|
| **CSV** | `.csv` | Texto plano | Universal, ligero, fácil de leer | Sin tipos, sin esquema, sin jerarquía |
| **JSON** | `.json` | Texto (semiestructurado) | Flexible, jerárquico, muy usado en APIs | Puede ser irregular, más voluminoso que CSV |
| **XML** | `.xml` | Texto (semiestructurado) | Muy usado en sistemas heredados, autodescriptivo | Verboso, parseo más complejo |
| **Excel** | `.xlsx` | Binario | Familiar, admite múltiples hojas y formato | Propietario, no apto para grandes volúmenes |
| **SQL / SQLite** | `.db`, `.sqlite` | Relacional | Consultas potentes, integridad referencial | Requiere motor de BD, aprendizaje de SQL |

> 💡 **CSV y JSON** son los formatos más habituales en proyectos de ML. SQL es el estándar en entornos empresariales. XML aparece frecuentemente en datos heredados o industriales.

### 📌 Ejemplo 1 — Creación y lectura de un archivo CSV

In [ ]:
# Creamos un CSV de ejemplo con datos de sensores industriales
csv_contenido = """sensor_id,timestamp,temperatura,humedad,estado
S01,2026-06-01 08:00:00,22.3,55.1,OK
S02,2026-06-01 08:00:00,23.1,58.4,OK
S03,2026-06-01 08:00:00,,61.2,FALLO
S01,2026-06-01 08:05:00,22.5,55.3,OK
S02,2026-06-01 08:05:00,23.4,57.9,OK
S03,2026-06-01 08:05:00,21.8,60.8,OK
"""

# Guardamos el CSV en disco
with open('sensores.csv', 'w', encoding='utf-8') as f:
    f.write(csv_contenido)

print('Contenido del archivo sensores.csv:')
print(csv_contenido)

In [ ]:
# Cargamos el CSV con pandas
df_csv = pd.read_csv('sensores.csv')

print('DataFrame cargado desde CSV:')
print(df_csv)
print(f'\nTipos detectados automáticamente:')
print(df_csv.dtypes)
print(f'\n⚠️ Fíjate: timestamp es "object" (str), temperatura tiene un NaN (valor ausente).')

In [ ]:
# Opciones útiles de read_csv para controlar la carga
df_csv2 = pd.read_csv(
    'sensores.csv',
    parse_dates=['timestamp'],      # Convierte la columna a datetime
    dtype={'sensor_id': 'category', # Fuerza el tipo de columnas concretas
           'estado': 'category'}
)

print('DataFrame con tipos corregidos en la carga:')
print(df_csv2.dtypes)
print()
print(df_csv2)

### 📌 Ejemplo 2 — Creación y lectura de un archivo JSON

In [ ]:
# Creamos un JSON con datos de productos de un e-commerce
datos_json = [
    {"id": 1, "nombre": "Teclado mecánico",  "precio": 89.99,  "categorias": ["periféricos", "gaming"],  "stock": 15},
    {"id": 2, "nombre": "Monitor 27\"",       "precio": 349.00, "categorias": ["pantallas"],             "stock": 4},
    {"id": 3, "nombre": "Ratón inalámbrico", "precio": 45.50,  "categorias": ["periféricos"],           "stock": 0},
    {"id": 4, "nombre": "Hub USB-C",         "precio": 29.99,  "categorias": ["accesorios", "USB"],     "stock": 22}
]

# Guardamos el JSON en disco
with open('productos.json', 'w', encoding='utf-8') as f:
    json.dump(datos_json, f, ensure_ascii=False, indent=2)

# Cargamos con pandas
df_json = pd.read_json('productos.json')

print('DataFrame cargado desde JSON:')
print(df_json)
print(f'\nTipos:')
print(df_json.dtypes)
print('\n⚠️ La columna "categorias" contiene listas → no es escalar, requiere transformación.')

### 📌 Ejemplo 3 — Lectura de un archivo XML

In [ ]:
# Creamos un XML con datos de empleados (formato típico en sistemas heredados / SAP)
xml_contenido = """<?xml version="1.0" encoding="UTF-8"?>
<empleados>
  <empleado id="E001">
    <nombre>Ana García</nombre>
    <departamento>IT</departamento>
    <salario>32000</salario>
    <fecha_alta>2023-09-01</fecha_alta>
  </empleado>
  <empleado id="E002">
    <nombre>Luis Martínez</nombre>
    <departamento>Ventas</departamento>
    <salario>28500</salario>
    <fecha_alta>2022-03-15</fecha_alta>
  </empleado>
  <empleado id="E003">
    <nombre>María López</nombre>
    <departamento>IT</departamento>
    <salario>35000</salario>
    <fecha_alta>2021-06-20</fecha_alta>
  </empleado>
</empleados>
"""

with open('empleados.xml', 'w', encoding='utf-8') as f:
    f.write(xml_contenido)

# Opción 1: pandas puede leer XML directamente (desde pandas 1.3)
df_xml = pd.read_xml('empleados.xml', xpath='.//empleado')
print('DataFrame cargado desde XML con pd.read_xml():')
print(df_xml)
print(f'\nTipos: {df_xml.dtypes.to_dict()}')

### 📌 Ejemplo 4 — Lectura de un archivo Excel

In [ ]:
# Creamos un Excel con dos hojas: ventas y devoluciones
df_ventas = pd.DataFrame({
    'fecha':     pd.to_datetime(['2026-01-10', '2026-01-12', '2026-01-15', '2026-01-18']),
    'producto':  ['Teclado', 'Monitor', 'Ratón', 'Hub USB-C'],
    'unidades':  [3, 1, 5, 2],
    'importe':   [269.97, 349.00, 227.50, 59.98]
})

df_devoluciones = pd.DataFrame({
    'fecha':    pd.to_datetime(['2026-01-14', '2026-01-20']),
    'producto': ['Ratón', 'Monitor'],
    'motivo':   ['defecto de fábrica', 'no cumple expectativas']
})

# Guardamos el Excel con múltiples hojas
with pd.ExcelWriter('tienda.xlsx', engine='openpyxl') as writer:
    df_ventas.to_excel(writer, sheet_name='ventas', index=False)
    df_devoluciones.to_excel(writer, sheet_name='devoluciones', index=False)

print('Archivo Excel "tienda.xlsx" creado con 2 hojas.')

# Cargamos una hoja concreta
df_ventas_cargado = pd.read_excel('tienda.xlsx', sheet_name='ventas', parse_dates=['fecha'])
print('\nHoja "ventas" cargada:')
print(df_ventas_cargado)

# Podemos cargar todas las hojas a la vez con sheet_name=None → devuelve un dict
hojas = pd.read_excel('tienda.xlsx', sheet_name=None)
print(f'\nHojas disponibles en el Excel: {list(hojas.keys())}')

### 📌 Ejemplo 5 — Lectura desde una base de datos SQLite

In [ ]:
# Creamos una base de datos SQLite con dos tablas relacionadas
conn = sqlite3.connect('tienda.db')
cursor = conn.cursor()

# Tabla de clientes
cursor.execute('''
    CREATE TABLE IF NOT EXISTS clientes (
        id       INTEGER PRIMARY KEY,
        nombre   TEXT NOT NULL,
        ciudad   TEXT,
        premium  INTEGER DEFAULT 0
    )
''')

# Tabla de pedidos
cursor.execute('''
    CREATE TABLE IF NOT EXISTS pedidos (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER REFERENCES clientes(id),
        producto    TEXT,
        importe     REAL,
        fecha       TEXT
    )
''')

# Insertamos datos de ejemplo
cursor.executemany('INSERT OR IGNORE INTO clientes VALUES (?,?,?,?)', [
    (1, 'Ana García',     'Sevilla', 1),
    (2, 'Luis Martínez',  'Madrid',  0),
    (3, 'María López',    'Málaga',  1),
])

cursor.executemany('INSERT OR IGNORE INTO pedidos VALUES (?,?,?,?,?)', [
    (1, 1, 'Teclado',  89.99,  '2026-01-10'),
    (2, 1, 'Monitor',  349.00, '2026-01-12'),
    (3, 2, 'Ratón',    45.50,  '2026-01-15'),
    (4, 3, 'Hub USB-C',29.99,  '2026-01-18'),
    (5, 2, 'Teclado',  89.99,  '2026-01-20'),
])

conn.commit()
print('Base de datos "tienda.db" creada con tablas: clientes, pedidos.')

In [ ]:
# Cargamos datos desde SQLite con pd.read_sql()
# Esto funciona de la misma forma con cualquier base de datos (PostgreSQL, MySQL, etc.)
# solo cambia la cadena de conexión

# Consulta simple: todos los clientes
df_clientes = pd.read_sql('SELECT * FROM clientes', conn)
print('Tabla clientes:')
print(df_clientes)

# Consulta con JOIN: pedidos con nombre del cliente
query_join = '''
    SELECT p.id AS pedido_id,
           c.nombre AS cliente,
           c.ciudad,
           p.producto,
           p.importe,
           p.fecha
    FROM pedidos p
    JOIN clientes c ON p.cliente_id = c.id
    ORDER BY p.fecha
'''
df_pedidos = pd.read_sql(query_join, conn, parse_dates=['fecha'])
print('\nPedidos con datos de cliente (JOIN):')
print(df_pedidos)

conn.close()

---
### 🔧 Ejercicio 1 — Elige el formato adecuado

Para cada situación, indica qué formato de almacenamiento usarías y por qué.

In [ ]:
# Completa el diccionario con tus respuestas.
# Valores posibles: 'CSV', 'JSON', 'XML', 'Excel', 'SQL'

situaciones = [
    'Intercambiar un dataset de 500.000 registros con un equipo externo sin acceso a BD',
    'Recibir datos de pedidos desde la API REST de un proveedor de logística',
    'Almacenar el historial de transacciones bancarias con relaciones entre clientes y cuentas',
    'Distribuir un informe de ventas mensual a un departamento comercial no técnico',
    'Leer configuraciones de máquinas de una fábrica que exporta datos en formato estándar ISO',
]

respuestas = {
    situaciones[0]: '???',   # Escribe tu respuesta
    situaciones[1]: '???',
    situaciones[2]: '???',
    situaciones[3]: '???',
    situaciones[4]: '???',
}

for situacion, respuesta in respuestas.items():
    print(f'{respuesta:8s} → {situacion}')

---
## 1.4 Carga de datos con Python/pandas

pandas proporciona una familia de funciones `read_*` que permiten cargar datos desde casi cualquier fuente. Todas devuelven un `DataFrame`.

```
pd.read_csv(ruta, sep=',', encoding='utf-8', parse_dates=[...], dtype={...})
pd.read_json(ruta)                         → JSON plano o lista de registros
pd.read_xml(ruta, xpath='.//elemento')     → XML con ruta XPath
pd.read_excel(ruta, sheet_name='hoja')     → Excel, una o varias hojas
pd.read_sql(query, conexión)               → cualquier BD compatible con SQLAlchemy
```

### Parámetros comunes y problemas frecuentes

| Parámetro | Función | Cuándo usarlo |
|-----------|---------|---------------|
| `sep` | Separador de campos | CSV con `;` o `\t` en lugar de `,` |
| `encoding` | Codificación del archivo | Archivos con tildes o ñ (usa `'utf-8'` o `'latin-1'`) |
| `parse_dates` | Convierte columnas a `datetime` | Columnas de fecha cargadas como texto |
| `dtype` | Fuerza el tipo de columnas | Evitar que pandas infiera tipos incorrectos |
| `na_values` | Define cadenas que se tratan como NaN | Valores como `'N/A'`, `'-'`, `'?'` |
| `nrows` | Número de filas a leer | Previsualizar un archivo enorme sin cargarlo entero |
| `usecols` | Columnas a cargar | Cargar solo las columnas necesarias para ahorrar memoria |

### 📌 Ejemplo 6 — Problemas comunes en la carga de CSV

In [ ]:
# CSV con problemas habituales: separador distinto, codificación, valores nulos no estándar
csv_problematico = """nombre;edad;salario;ciudad;activo
Ana García;23;28.000,00 €;Sevilla;Sí
Luis Martínez;35;N/A;Madrid;No
María López;28;32.500,00 €;Málaga;-
Carlos Ruiz;;45.000,00 €;Barcelona;Sí
"""

with open('empleados_problemas.csv', 'w', encoding='utf-8') as f:
    f.write(csv_problematico)

# Carga sin ningún parámetro adicional → resultado incorrecto
print('❌ Carga sin parámetros (resultado incorrecto):')
df_mal = pd.read_csv('empleados_problemas.csv')
print(df_mal)
print(f'\nColumnas detectadas: {df_mal.columns.tolist()}')

In [ ]:
# Carga correcta con los parámetros adecuados
df_bien = pd.read_csv(
    'empleados_problemas.csv',
    sep=';',                          # Separador correcto
    encoding='utf-8',
    na_values=['N/A', '-', ''],       # Valores que deben tratarse como NaN
    dtype={'nombre': 'string',
           'ciudad': 'category'}
)

print('✅ Carga correcta:')
print(df_bien)
print(f'\nTipos: {df_bien.dtypes.to_dict()}')
print(f'\n⚠️ Quedan por resolver: la columna "salario" (tiene formato europeo con puntos y €)')
print(f'   y la columna "activo" ("Sí"/"No" en lugar de True/False).')

### 📌 Ejemplo 7 — Cargar solo lo necesario de un archivo grande

In [ ]:
# Simulamos un archivo CSV grande (10.000 filas)
import numpy as np
np.random.seed(42)

n = 10_000
df_grande = pd.DataFrame({
    'id':          range(n),
    'temperatura': np.random.normal(22, 3, n).round(2),
    'humedad':     np.random.normal(55, 10, n).round(1),
    'presion':     np.random.normal(1013, 5, n).round(1),
    'sensor':      np.random.choice(['S01', 'S02', 'S03', 'S04'], n),
    'estado':      np.random.choice(['OK', 'OK', 'OK', 'WARN', 'FALLO'], n)
})
df_grande.to_csv('sensores_grande.csv', index=False)

print(f'Archivo "sensores_grande.csv" creado con {n:,} filas.')

# Técnica 1: leer solo las primeras filas para inspeccionar
df_preview = pd.read_csv('sensores_grande.csv', nrows=5)
print('\nPrimeras 5 filas (nrows=5):')
print(df_preview)

# Técnica 2: leer solo las columnas necesarias
df_reducido = pd.read_csv('sensores_grande.csv', usecols=['id', 'temperatura', 'estado'])
print(f'\nDataFrame con solo 3 columnas (usecols): {df_reducido.shape[0]:,} filas × {df_reducido.shape[1]} columnas')
print(df_reducido.head())

---
### 🔧 Ejercicio 2 — Carga y corrección de un CSV problemático

El archivo `empleados_problemas.csv` cargado en el Ejemplo 6 dejaba dos columnas sin corregir. Completa las tareas:

In [ ]:
# Partimos del DataFrame df_bien del Ejemplo 6
df_ej2 = df_bien.copy()

# TAREA 1: Limpia la columna 'salario'
# Pista: elimina los puntos de miles, la coma decimal y el símbolo €, luego convierte a float
# df_ej2['salario'] = df_ej2['salario'].str.replace(...).str.replace(...).str.replace(...).astype(float)

# TAREA 2: Convierte la columna 'activo' a booleano
# Pista: usa .map({'Sí': True, 'No': False})
# df_ej2['activo'] = ...

# TAREA 3: Verifica los tipos finales
# print(df_ej2.dtypes)
# print(df_ej2)

print('Completa las tareas comentadas arriba.')

---
## 1.5 Repositorios públicos de datos para ML

Antes de recopilar datos propios, conviene explorar los repositorios públicos disponibles. Son fuentes esenciales para practicar, investigar y desarrollar modelos.

| Repositorio | URL | Características |
|-------------|-----|-----------------|
| **Kaggle** | [kaggle.com/datasets](https://www.kaggle.com/datasets) | Miles de datasets con notebooks de ejemplo. Competiciones de ML. |
| **UCI ML Repository** | [archive.ics.uci.edu](https://archive.ics.uci.edu) | Clásico académico. Datasets muy usados en publicaciones científicas. |
| **Hugging Face** | [huggingface.co/datasets](https://huggingface.co/datasets) | Orientado a NLP, visión e IA generativa. API Python sencilla. |
| **OpenML** | [openml.org](https://www.openml.org) | Benchmarking reproducible. Integración con scikit-learn. |
| **Google Dataset Search** | [datasetsearch.research.google](https://datasetsearch.research.google.com) | Buscador de datasets de todo tipo. |

> 💡 **Kaggle** es el más popular para competiciones y proyectos de portafolio. **UCI** es la referencia académica clásica. **Hugging Face Datasets** es la opción más moderna y con mejor API para Python.

### 📌 Ejemplo 8 — Carga directa desde una URL (CSV público)

La forma más sencilla de usar datos públicos es cargarlos directamente desde su URL.

In [ ]:
# pandas puede leer directamente desde una URL
# Usamos el dataset Iris del UCI ML Repository (espejo en GitHub)

url_iris = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'

try:
    df_iris_url = pd.read_csv(url_iris)
    print(f'Dataset Iris cargado desde URL: {df_iris_url.shape[0]} filas × {df_iris_url.shape[1]} columnas')
    print(df_iris_url.head())
    print(f'\nEspecies: {df_iris_url["species"].unique()}')
except Exception as e:
    print(f'❌ Error al cargar desde URL (¿sin conexión?): {e}')
    print('→ En ese caso, usa: import seaborn as sns; df_iris_url = sns.load_dataset("iris")')

### 📌 Ejemplo 9 — Carga con la API de Hugging Face Datasets

La librería `datasets` de Hugging Face permite cargar miles de conjuntos de datos con una sola línea.

In [ ]:
# Instalamos la librería si no está disponible
# Solo es necesario ejecutar esto una vez
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'datasets', '-q'],
    capture_output=True, text=True
)
print('✅ datasets instalado.' if result.returncode == 0 else f'❌ Error: {result.stderr}')

In [ ]:
# Carga del dataset "rotten_tomatoes" desde Hugging Face
# (dataset de reseñas de películas para clasificación de sentimientos)
try:
    from datasets import load_dataset

    # Cargamos solo el split de entrenamiento
    ds = load_dataset('rotten_tomatoes', split='train', trust_remote_code=False)

    print(f'Dataset: rotten_tomatoes (train)')
    print(f'Número de ejemplos: {len(ds):,}')
    print(f'Columnas: {ds.column_names}')
    print(f'\nPrimeros 3 ejemplos:')
    for i in range(3):
        etiqueta = '✅ positiva' if ds[i]['label'] == 1 else '❌ negativa'
        print(f'  [{etiqueta}] {ds[i]["text"][:80]}...')

    # Convertimos a DataFrame de pandas para trabajar con él como siempre
    df_hf = ds.to_pandas()
    print(f'\nConvertido a DataFrame pandas: {df_hf.shape}')
    print(df_hf.head())

except Exception as e:
    print(f'❌ Error al cargar desde Hugging Face (¿sin conexión?): {e}')

### 📌 Ejemplo 10 — Exploración inicial tras la carga

Sea cual sea la fuente y el formato, los primeros pasos tras cargar un dataset siempre son los mismos.

In [ ]:
# Usamos el dataset de sensores creado en el Ejemplo 7
df = pd.read_csv('sensores_grande.csv')

print('=' * 50)
print('EXPLORACIÓN INICIAL DEL DATASET')
print('=' * 50)

# 1. Forma del dataset
print(f'\n📐 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas')

# 2. Primeras filas
print('\n📋 Primeras filas:')
print(df.head())

# 3. Tipos de datos
print('\n🔡 Tipos de datos (dtypes):')
print(df.dtypes)

# 4. Resumen con info()
print('\nℹ️ Resumen con info():')
df.info()

# 5. Estadísticos básicos de variables numéricas
print('\n📊 Estadísticos básicos (describe):')
print(df.describe().round(2))

---
### 🔧 Ejercicio 3 — Carga y exploración inicial de un dataset público

Elige uno de los siguientes datasets de Kaggle/UCI y realiza las tareas indicadas.

**Opción A** — Titanic (kaggle): `https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv`  
**Opción B** — Wine Quality (UCI): `https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv`  
**Opción C** — Cualquier dataset que elijas de Kaggle o UCI (documenta la fuente)

In [ ]:
# TAREA 1: Carga el dataset elegido (desde URL o archivo local)
# url = '...'
# df_ej3 = pd.read_csv(url)   # o pd.read_csv(url, sep=';') si es Wine Quality

# TAREA 2: Muestra las dimensiones del dataset
# print(f'Dimensiones: {df_ej3.shape}')

# TAREA 3: Muestra los tipos de datos con dtypes
# print(df_ej3.dtypes)

# TAREA 4: Muestra un resumen completo con info()
# df_ej3.info()

# TAREA 5: Muestra los estadísticos básicos con describe()
# print(df_ej3.describe())

# TAREA 6 (reflexión): ¿Qué tipo estadístico real tiene cada variable?
#   Anótalo como comentario o en una celda Markdown debajo.

print('Completa las tareas comentadas arriba. Elige una de las opciones A, B o C.')

---
### 🔧 Ejercicio 4 — Exportación a distintos formatos

Carga el dataset de sensores (`sensores_grande.csv`) y guárdalo en tres formatos distintos.

In [ ]:
# Cargamos el dataset de sensores del Ejemplo 7
df_sensores = pd.read_csv('sensores_grande.csv')

# Filtramos solo los registros de los sensores S01 y S02 con estado OK
df_filtrado = df_sensores[
    (df_sensores['sensor'].isin(['S01', 'S02'])) &
    (df_sensores['estado'] == 'OK')
].head(100)   # Solo primeros 100 para el ejercicio

print(f'Dataset filtrado: {df_filtrado.shape}')

# TAREA 1: Guarda df_filtrado como CSV
# df_filtrado.to_csv('sensores_filtrado.csv', index=False)

# TAREA 2: Guarda df_filtrado como JSON
# df_filtrado.to_json('sensores_filtrado.json', orient='records', indent=2)

# TAREA 3: Guarda df_filtrado como Excel
# df_filtrado.to_excel('sensores_filtrado.xlsx', index=False)

# TAREA 4: Verifica que los archivos se han creado correctamente
# y que el contenido es el esperado cargándolos de nuevo

print('\nCompleta las tareas comentadas arriba.')

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| Formatos de almacenamiento | CSV (universal), JSON (APIs/semiestructurado), XML (sistemas heredados), Excel (usuarios finales), SQL (BD relacionales). |
| Familia `read_*` de pandas | `read_csv`, `read_json`, `read_xml`, `read_excel`, `read_sql`. Todas devuelven un DataFrame. |
| Parámetros clave de carga | `sep`, `encoding`, `parse_dates`, `dtype`, `na_values`, `usecols`, `nrows`. |
| Repositorios públicos | Kaggle, UCI ML Repository, Hugging Face Datasets, OpenML. pandas puede leer directamente desde URL. |
| Exploración inicial | Siempre aplicar `shape`, `head()`, `dtypes`, `info()` y `describe()` tras cargar cualquier dataset. |

### 🔜 Próxima sesión
**Bloque 2 — Sesión 1:** Análisis Exploratorio de Datos (AED): inspección inicial y descripción estadística de variables numéricas.